In [10]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import os
from hmmlearn.hmm import GaussianHMM
from dataclasses import dataclass, asdict


In [56]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def download(self, indices:str, start:str, end:str, interval:str):
    data = {}
    for ticker in indices:
        clean_ticker = ticker.strip()
        try:
            tmp_data = yf.Ticker(clean_ticker) \
                          .history(
                              start=start,
                              end=end,
                              interval=interval
                          )["Close"]

            if tmp_data.empty:
                print(f"Warning: No data returned for {clean_ticker}")
                continue
            tmp_data.index = pd.to_datetime(tmp_data.index)\
                                    .tz_localize(None)\
                                    .normalize()
            tmp_data = tmp_data[~tmp_data.index.duplicated(keep='last')]
            data[clean_ticker] = tmp_data
        except Exception as e:
            print(f"Failed to download {clean_ticker}: {e}")

    if not data:
        raise ValueError(
            "No data was successfully downloaded  \
              for any of the provided tickers."
        )
    df = pd.concat(data, axis=1)
    return df.sort_index()

  def get_data(
      self,
      tickers:list,
      start:str,
      end:str,
      mkt_sectors:list[str],
      interval:str="1d"
  ):
    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"
    benchmark_path = f"benchmark_{start}_{end}.parquet"

    if not os.path.exists(df_path) or not os.path.exists(benchmark_path):
      df = self.download(tickers, start, end, interval)
      bench_data = self.download(benchmark, start, end, interval)

      df.to_parquet(df_path)
      bench_data.to_parquet(benchmark_path)

    else:
      df = pd.read_parquet(df_path)
      bench_data= pd.read_parquet(benchmark_path)

    returns_raw = df.pct_change().dropna()
    benchmark = bench_data.pct_change().dropna()

    return returns_raw, benchmark

  def get_market_data(self, ticker, start, end, interval="1d"):
    return yf.Ticker(ticker).history(start=start, end=end, interval=interval)

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [69]:
class HMMClassifier:
  def __init__(self, hmm_params, debug=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug
    hmm_params = asdict(hmm_params)
    self.hmm = GaussianHMM(**asdict(hmm_params))
    self.trained = False

  def fit(self, X):
    self.trained = True
    return self.hmm.fit(X)

  def get_regime_probs(self, X):
    if not self.trained:
      raise ValueError("HMM Classifier not trained")

    return self.hmm.predict_proba(X)

In [ ]:
class RegimeClassifier(HMMClassifier):
    def __init__(self, debug=False, **kwargs):
      super().__init__(debug=debug, **kwargs)
      self.debug = debug

    def z_score_norm(self, X):
      X_cent = X - np.mean(X, axis=0)
      return X_cent / np.std(X_cent, axis=0, ddof=0)

    def z_score(self, x):
      mu = np.mean(x)
      sigma = np.std(x, ddof=0)
      return (x - mu) / sigma if sigma > 0 else np.zeros_like(x)

    def _apply_ledoit_wolf(self, r_norm, gram_sample, T, n_assets):
      mean_mkt_corr = (np.sum(gram_sample) - n_assets) / (n_assets * (n_assets - 1))
      target = np.full((n_assets, n_assets), mean_mkt_corr)
      np.fill_diagonal(target, 1)

      noise_mtx = 0.0
      for t in range(T):
        r_t = r_norm[t, :].reshape(-1, 1)
        sample_t_mtx = r_t @ r_t.T
        noise_mtx += np.sum((sample_t_mtx - gram_sample) ** 2)

      noise = noise_mtx / (T ** 2)
      dist  = np.sum((gram_sample - target) ** 2)

      if dist == 0:
        return gram_sample

      shrinkage_intensity = np.clip(noise / dist, 0.0, 1.0)

      if self.debug:
        print(f"Ledoit-Wolf shrinkage intensity: {shrinkage_intensity:.4f}")

      return shrinkage_intensity * target + (1 - shrinkage_intensity) * gram_sample

    def calculate_spec_ent(self, eig_vals):
      regime_probs = eig_vals / np.sum(eig_vals)
      spec_ent = -np.sum(regime_probs * np.log(regime_probs))
      return spec_ent

    def _get_eff_dim(self, r_norm, T):
      G = self._get_gram_mtx(r_norm, T)
      n_assets = G.shape[0]
      G_cond   = self._apply_ledoit_wolf(r_norm, G, T, n_assets)
      eig_vals = np.clip(np.linalg.eigvalsh(G_cond), a_min=1e-12, a_max=None)
      spec_ent = self.calculate_spec_ent(eig_vals)
      return np.exp(spec_ent)

    def _get_gram_mtx(self, r, T):
      return (1 / T) * r.T @ r

    def garman_class_vol(self, market, lambda_param=0.94):
      ln_CO = np.log(market["Close"] / market["Open"])
      ln_HL = np.log(market["High"] / market["Low"])
      gk_var = 0.5 * ln_HL ** 2 - (2 * np.log(2) - 1) * ln_CO ** 2

      T = len(gk_var)
      smoothed_variance    = np.zeros(T)
      smoothed_variance[0] = gk_var.iloc[0]
      for t in range(1, T):
        smoothed_variance[t] = (
          lambda_param * smoothed_variance[t - 1]
          + (1 - lambda_param) * gk_var.iloc[t]
        )
      return np.sqrt(smoothed_variance * 252)

    def _process_data(self, returns, market):
      r_log = np.log(1 + returns)
      T = r_log.shape[0]
      vol = self.garman_class_vol(market)

      r_norm = self.z_score_norm(r_log)
      eff_dim = self._get_eff_dim(r_norm, T)

      eff_dim_series = np.full(T, eff_dim)

      eff_dim_norm = self.z_score(eff_dim_series)
      vol_norm = self.z_score(vol)

      X = np.column_stack((eff_dim_norm, vol_norm))
      return X

    def train_classifier(self, returns, market):
      X = self._process_data(returns, market)
      self.fit(X)

    def get_regime_probs(self, returns, market):
      X = self._process_data(returns, market)
      return super().get_regime_probs(X)

    def score_bic(self, X):
      T, n_features  = X.shape
      n_components   = self.model.n_components
      log_likelihood = self.model.score(X) * T

      k_transition = n_components * (n_components - 1)
      k_means = n_components * n_features
      k_vars = n_components * n_features
      k = k_transition + k_means + k_vars

      bic = k * np.log(T) -2 * log_likelihood
      return bic


    def expected_regime_duration(self):
      diag = np.diag(self.model.transmat_)
      diag = np.clip(diag, 1e-10, 1 - 1e-10)
      return 1.0 / (1.0 - diag)


In [ ]:
class Backtest(RegimeClassifier):
    def __init__(self, debug=False, **kwargs):
      super().__init__(debug=debug, **kwargs)

    def get_data(
        self,
        tickers: list,
        start: str,
        end: str,
        market_proxy: list[str],
        interval: str = "1d",
        market_ticker: str = "SPY"
    ):
      market_ohlc = self.get_market_data(market_ticker, start, end)
      data, sectors = DataStore().get_data(
        tickers=tickers,
        start=start,
        end=end,
        interval=interval,
        market_proxy=market_proxy
      )
      return data, sectors, market_ohlc

    def run_validation(
        self,
        returns: pd.DataFrame,
        market: pd.DataFrame,
        train_months: int = 12,
        test_months: int = 6,
        step_months: int = 6
    ):
      results = []
      index = returns.index

      DAYS_PER_MONTH = 21
      train_days = train_months * DAYS_PER_MONTH
      test_days = test_months * DAYS_PER_MONTH
      step_days = step_months * DAYS_PER_MONTH

      total_days = len(returns)
      fold = 0

      print(f"\n{'='*62}")
      print(f"  Walk-forward validation  |  "
            f"train={train_months}M  test={test_months}M  step={step_months}M")
      print(f"{'='*62}\n")

      for train_start in range(0, total_days - train_days - test_days + 1, step_days):
        train_end = train_start + train_days
        test_end = min(train_end + test_days, total_days)

        train_ret = returns.iloc[train_start:train_end]
        test_ret = returns.iloc[train_end:test_end]
        train_mkt = market.iloc[train_start:train_end]
        test_mkt = market.iloc[train_end:test_end]

        if len(test_ret) == 0:
          break

        fold += 1
        train_start_dt = index[train_start].strftime("%Y-%m-%d")
        train_end_dt = index[train_end - 1].strftime("%Y-%m-%d")
        test_end_dt = index[test_end - 1].strftime("%Y-%m-%d")

        print(f"Fold {fold:02d} | "
              f"train [{train_start_dt} => {train_end_dt}] | "
              f"test  [{index[train_end].strftime('%Y-%m-%d')} => {test_end_dt}]")

        self.train_classifier(train_ret, train_mkt)

        X_train = self._process_data(train_ret, train_mkt)
        train_bic = self.score_bic(X_train)
        train_durations = self.expected_regime_duration()

        print(f"  [train] BIC: {train_bic:.2f}")
        for s, dur in enumerate(train_durations):
          print(f"          Regime {s} expected duration: {dur:.1f} trading days "
                f"({dur/DAYS_PER_MONTH:.1f} months)")

        X_test = self._process_data(test_ret, test_mkt)
        test_ll = self.model.score(X_test) * len(X_test)
        test_bic = self.score_bic(X_test)
        regime_probs = self.get_regime_probs(test_ret, test_mkt)
        dominant_regime = np.argmax(regime_probs.mean(axis=0))

        print(f"  [test]  Log-likelihood: {test_ll:.2f} | "
              f"BIC: {test_bic:.2f} | "
              f"Dominant regime: {dominant_regime}")
        print()

        results.append({
          "fold": fold,
          "train_start": train_start_dt,
          "train_end": train_end_dt,
          "test_end": test_end_dt,
          "train_bic": train_bic,
          "test_bic": test_bic,
          "test_log_lik": test_ll,
          "dominant_regime": dominant_regime,
          "durations": train_durations.tolist(),
        })

      summary = pd.DataFrame(results).set_index("fold")

      print(f"\n{'='*62}")
      print("Validation summary")
      print(f"{'='*62}")
      print(summary[["train_bic", "test_bic", "test_log_lik", "dominant_regime"]]
            .to_string())
      print(f"\nMean test BIC       : {summary['test_bic'].mean():.2f}")
      print(f"Mean test log-lik   : {summary['test_log_lik'].mean():.2f}")
      print(f"{'='*62}\n")

      return summary

In [54]:
@dataclass
class HMMParams:
  n_components: int = 2
  n_iter: int = 50
  tol: float = 1e-4
  init_params: str = "stmc"
  cov_type: str = "diag"